In [1]:
# ============================================================
# 1. Imports & Setup
# ============================================================
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna

# Precision Settings
TORCH_DTYPE = torch.float32
NP_DTYPE = np.float32

torch.set_default_dtype(TORCH_DTYPE)
warnings.filterwarnings("ignore", category=FutureWarning)

# Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")
if device.type == "cuda":
    print(f"CUDA device: {torch.cuda.current_device()} - {torch.cuda.get_device_name(0)}")

# ============================================================
# 2. Reproducibility & Data Loading
# ============================================================
base_seed = 2025
np.random.seed(base_seed)
torch.manual_seed(base_seed)
torch.cuda.manual_seed_all(base_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load Clean Reference
data_clean = np.load("lorenz63_noise_0.npy", allow_pickle=True).item()
l63 = data_clean["data"]
if l63.shape[0] == 3:  # Force (T, d) layout standard
    l63 = l63.T
l63 = l63.astype(NP_DTYPE)

# Load Noisy Dataset
data_noisy = np.load("lorenz63_noise_10.npy", allow_pickle=True).item()
X_noisy_dataset = data_noisy["data"]
if X_noisy_dataset.shape[0] == 3:
    X_noisy_dataset = X_noisy_dataset.T
X_noisy_dataset = X_noisy_dataset.astype(NP_DTYPE)

X = torch.tensor(X_noisy_dataset, dtype=TORCH_DTYPE) 
X_true = torch.tensor(l63, dtype=TORCH_DTYPE)         

# Dataset Splits 
warmup_len, train_len, val_len, test_len = 100, 3900, 500, 500

X_warmup = X[:warmup_len].to(device)
X_train_clean = X[warmup_len:warmup_len + train_len]
X_val  = X[warmup_len + train_len : warmup_len + train_len + val_len].to(device)
X_test = X[warmup_len + train_len + val_len : warmup_len + train_len + val_len + test_len].to(device)

X_val_true  = X_true[warmup_len + train_len : warmup_len + train_len + val_len].to(device)
X_test_true = X_true[warmup_len + train_len + val_len : warmup_len + train_len + val_len + test_len].to(device)

# Training Regularization Noise Setup
train_noise_scale = 0.0  
signal_std = X_train_clean.std(dim=0, keepdim=True).numpy()
train_noise = np.random.normal(0.0, train_noise_scale * signal_std, X_train_clean.shape)
X_train = torch.tensor((X_train_clean.numpy() + train_noise).astype(NP_DTYPE), device=device)

# Structural Parameters
d = 3
horizons = [25, 50, 75, 100]
num_windows = 5   

# ============================================================
# 3. Model Architecture & Helpers
# ============================================================
class FeatureMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim, dtype=TORCH_DTYPE),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim, dtype=TORCH_DTYPE)
        )

    def forward(self, x):
        return self.net(x)


class AdaptiveNVARModel(nn.Module):
    def __init__(self, dk, m, d_out, hidden_dim):
        super().__init__()
        self.mlp = FeatureMLP(dk, hidden_dim, m)
        self.readout = nn.Linear(dk + m, d_out, bias=True, dtype=TORCH_DTYPE)

    def forward(self, H_lin):
        H_nn = self.mlp(H_lin)
        H_total = torch.cat([H_lin, H_nn], dim=1)
        return self.readout(H_total)


def construct_H_lin(X_tensor, k):
    """Build delay vectors: [x(t), x(t-1), ..., x(t-k+1)]"""
    T = X_tensor.shape[0]
    H = []
    for t in range(k - 1, T - 1):
        delays = [X_tensor[t - delay] for delay in range(k)]
        H.append(torch.cat(delays, dim=0))
    return torch.stack(H)


def init_weights_stable(m):
    if isinstance(m, nn.Linear):
        if m.bias is None:
            nn.init.normal_(m.weight, mean=0.0, std=1e-4)
        else:
            nn.init.xavier_normal_(m.weight, gain=nn.init.calculate_gain('tanh'))
            nn.init.zeros_(m.bias)


# ============================================================
# 4. Training Engine (Optimized for Deep Convergence)
# ============================================================
def train_joint_model(
    X_input, k, m, hidden_dim=200,
    lr_adam=1e-4, max_epochs_adam=15000,   # Increased max epochs slightly
    adam_patience=1500, tolerance=1e-8,    # Much stricter tolerance & higher patience
    device_target=None
):
    dev = device_target or device
    X_local = X_input.to(device=dev, dtype=TORCH_DTYPE)

    H_lin = construct_H_lin(X_local, k)       
    Y = X_local[k:] - X_local[k-1:-1]         
    dk = H_lin.shape[1]
    d_out = Y.shape[1]

    model = AdaptiveNVARModel(dk, m, d_out, hidden_dim).to(dev)
    model.apply(init_weights_stable)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_adam, weight_decay=1e-4)
    
    # Increased patience so it doesn't decay the learning rate prematurely
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=100  
    )

    best_loss = float("inf")
    epochs_no_improve = 0
    in_memory_state = None
    
    for epoch in range(max_epochs_adam):
        model.train()
        Y_hat = model(H_lin)
        loss = F.mse_loss(Y_hat, Y)

        optimizer.zero_grad()
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step(loss.item())

        # Stricter tracking down to 1e-8 changes
        if best_loss - loss.item() > tolerance:
            best_loss = loss.item()
            epochs_no_improve = 0
            in_memory_state = {k_v: v.cpu().clone() for k_v, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= adam_patience:
            break

    if in_memory_state is not None:
        model.load_state_dict({k_v: v.to(dev) for k_v, v in in_memory_state.items()})

    return model


# ============================================================
# 5. Core Evaluation Pipeline Function (Fixed)
# ============================================================
def evaluate_model(
    model, k, X_true_target, X_history, num_w, h_list, dev
):
    """
    Evaluates rolling windows over a target block using localized indices.
    
    X_true_target : Clean ground truth array for the evaluation window (val or test).
    X_history     : Array containing exactly [k historical points + evaluation block points].
    """
    window_size = int(len(X_true_target) // num_w)
    horizon_rmses = {h: [] for h in h_list}
    model.eval()

    for w in range(num_w):
        start = w * window_size
        end   = (w + 1) * window_size

        X_true_window = X_true_target[start:end].to(device=dev, dtype=TORCH_DTYPE)

        # Grab the immediate history segment preceding the rollout window start
        init_start = start
        init_end   = start + k
        X_init = X_history[init_start:init_end].to(device=dev, dtype=TORCH_DTYPE)

        x_t = [x.clone() for x in X_init.unbind(0)]
        H_lin = torch.cat([x_t[-1 - delay] for delay in range(k)], dim=0).unsqueeze(0)

        predictions = []
        for _ in range(window_size):
            with torch.no_grad():
                delta_x = model(H_lin)
            
            x_next = x_t[-1] + delta_x.squeeze(0)
            predictions.append(x_next)

            x_t = x_t[1:] + [x_next]
            H_lin = torch.cat([x_t[-1 - delay] for delay in range(k)], dim=0).unsqueeze(0)

        predictions_np = torch.stack(predictions).cpu().numpy().astype(NP_DTYPE)
        y_true_np = X_true_window.cpu().numpy().astype(NP_DTYPE)

        for h in h_list:
            if h <= window_size:
                rmse = np.sqrt(np.mean((y_true_np[:h] - predictions_np[:h]) ** 2))
                horizon_rmses[h].append(rmse)

    return {h: np.mean(horizon_rmses[h]) for h in h_list}


# ============================================================
# 6. Optuna Target Objective 
# ============================================================
def objective(trial):
    k_suggest = trial.suggest_categorical("k", [2, 5, 10, 20, 30, 40, 50])
    hidden_dim_suggest = trial.suggest_categorical("hidden_dim", [8, 32, 128, 256, 512])
    lr_adam_suggest = trial.suggest_categorical("lr_adam", [1e-4, 5e-4, 1e-3])
    
    m_suggest = 10 * (d * k_suggest)
    
    # Isolate history window specifically for validation phase
    X_val_history = X[warmup_len + train_len - k_suggest : warmup_len + train_len + val_len]
    
    # Evaluate across multiple initial seeds to verify parameter stability
    validation_scores = []
    for eval_run in range(1): 
        run_seed = base_seed + eval_run
        torch.manual_seed(run_seed)
        torch.cuda.manual_seed_all(run_seed)
        
        model = train_joint_model(
            X_train, k_suggest, m_suggest, hidden_dim_suggest, lr_adam_suggest, device_target=device
        )
        rmse_by_horizon = evaluate_model(
            model, k_suggest, X_val_true, X_val_history, num_windows, horizons, device
        )
        validation_scores.append(rmse_by_horizon[100])
        
    return np.mean(validation_scores)


# ============================================================
# 7. Main Automated Execution Workflow
# ============================================================
if __name__ == "__main__":
    # --- PHASE 1: HYPERPARAMETER TUNING ---
    print("Starting Automated Optuna Parameter Search Stage...\n")
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=base_seed)
    )
    study.optimize(objective, n_trials=10, show_progress_bar=True)

    best_trial = study.best_trial
    best_params = best_trial.params

    print("\n" + "=" * 60)
    print("BEST CONFIGURATION RETRIEVED AUTOMATICALLY")
    print("=" * 60)
    print(f"k          : {best_params['k']}")
    print(f"hidden_dim : {best_params['hidden_dim']}")
    print(f"lr_adam    : {best_params['lr_adam']:.6e}")
    print(f"Validation Target Cross-Seed Average RMSE@100: {best_trial.value:.6f}")

    # --- PHASE 2: AUTOMATED EXTRACTION FOR MULTI-RUN BENCHMARK ---
    print("\n=== Launching Final Test Benchmark Using Best Values ===")
    
    k_best = best_params['k']
    hd_best = best_params['hidden_dim']
    lr_best = best_params['lr_adam']
    m_best = 10 * (d * k_best)
    
    # Isolate history window specifically for test phase
    X_test_history = X[warmup_len + train_len + val_len - k_best : warmup_len + train_len + val_len + test_len]
    
    num_runs = 25
    all_run_rmses = {h: [] for h in horizons}

    for run in range(num_runs):
        run_seed = base_seed + run
        torch.manual_seed(run_seed)
        torch.cuda.manual_seed_all(run_seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        # Train model using optimized configurations
        model = train_joint_model(
            X_train, k=k_best, m=m_best, hidden_dim=hd_best,
            lr_adam=lr_best, device_target=device
        )
        
        # Run test evaluation with the cleanly sliced history block
        rmse_by_horizon = evaluate_model(
            model, k_best, X_test_true, X_test_history, num_windows, horizons, device
        )
        
        for h in horizons:
            all_run_rmses[h].append(rmse_by_horizon[h])
        print(f"Run {run+1:02d}/{num_runs} Complete. Seed: {run_seed} -> h100: {rmse_by_horizon[100]:.6f}")

    # --- PHASE 3: FINAL REPORT ---
    final_stats = {h: (np.mean(all_run_rmses[h]), np.std(all_run_rmses[h], ddof=1)) for h in horizons}

    print("\n" + "=" * 60)
    print("FINAL TEST EXTRAPOLATION SUMMARY (Over 25 Runs)")
    print("=" * 60)
    print(f"Parameters utilized: k={k_best}, hidden_dim={hd_best}, train_noise={train_noise_scale}")
    print("-" * 60)
    for h in horizons:
        mean, std = final_stats[h]
        print(f"Horizon {h:3d} steps: {mean:.6f} ± {std:.6f}")

/home/eric/.conda/envs/nvar/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device in use: cuda
CUDA device: 0 - A100-PCIE-40GB
(5001, 3)

=== Test Prediction over Multiple Runs (Adam-Only Framework) ===

--- Run 1/25 | Seed=2025 ---
Run RMSEs → h=25: 0.406057, h=50: 0.503649, h=75: 0.489369, h=100: 0.540148

--- Run 2/25 | Seed=2026 ---
Run RMSEs → h=25: 0.270068, h=50: 0.312117, h=75: 0.273934, h=100: 0.260231

--- Run 3/25 | Seed=2027 ---
Run RMSEs → h=25: 0.276938, h=50: 0.367810, h=75: 0.322971, h=100: 0.453096

--- Run 4/25 | Seed=2028 ---
Run RMSEs → h=25: 0.434854, h=50: 0.472989, h=75: 0.472673, h=100: 0.560074

--- Run 5/25 | Seed=2029 ---
Run RMSEs → h=25: 0.306397, h=50: 0.396930, h=75: 0.356931, h=100: 0.407786

--- Run 6/25 | Seed=2030 ---
Run RMSEs → h=25: 0.474816, h=50: 0.487022, h=75: 0.473114, h=100: 0.537318

--- Run 7/25 | Seed=2031 ---
Run RMSEs → h=25: 0.483004, h=50: 0.474535, h=75: 0.465208, h=100: 0.606198

--- Run 8/25 | Seed=2032 ---
Run RMSEs → h=25: 0.267934, h=50: 0.287357, h=75: 0.259024, h=100: 0.426458

--- Run 9/25 | Seed=203